# Corte 3, Punto 1: deduplicación y capas silver y gold

Por: Gabriel Rodriguez y Jonatan Palomares

**Subtareas:** T1.1 a T1.6 (diagnóstico de duplicados, silver, gold, tablas resumen, reconciliación y linaje).

Entrada: bronze. Salidas: `silver/`, `gold/` y `procesado/`.

Cada sección trae su subtarea, el código y sus criterios de aceptación. El linaje de datos de todo el flujo (T1.6) está al final y en `docs/articulacion_cortes.md`.

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [1]:
# Importación de librerías

import os
from time import perf_counter
from statistics import median
import random
import math
import json
import urllib.request
from IPython.display import display, Markdown

import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark.sql.types import DateType, TimestampType
from pyspark import StorageLevel

from pamd.config import (
    INPUT_CSV,
    BRONZE_PARQUET_UNPARTITIONED,
    BRONZE_PARQUET,
    QUARANTINE_PARQUET,
    STAGING_PARQUET,
    SILVER_PARQUET,
    SILVER_DESCARTADOS_PARQUET,
    GOLD_PARQUET,
    GOLD_EXCLUIDOS_PARQUET,
    PROCESADO_ENTIDAD_PARQUET,
    PROCESADO_CONTRATISTA_PARQUET,
    PROCESADO_DEPARTAMENTO_PARQUET,
    PROCESADO_MES_PARQUET,
    WRITE_MODE,
    OUTPUT_PARTITIONS,
)
from pamd.spark_session import get_spark
from pamd.calidad import (
    UMBRAL_RAZON_PRECIO_BASE,
    UMBRAL_EXTREMO,
    UMBRAL_EXTREMO_SIN_BASE,
    marcar_atipicos,
    unificar_nit_entidad,
)

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()


## T1.1 Diagnóstico de duplicados y elección de la clave natural

El bronze conserva todas las filas válidas del CSV, pero un mismo proceso de contratación puede aparecer varias veces (nuevas publicaciones, lotes o varias adjudicaciones). Antes de deduplicar hay que medir cuánto se repite cada clave candidata y entender por qué, para elegir la clave que define **un contrato adjudicado**.

Claves candidatas:
- **Fila completa:** filas idénticas en todas las columnas (duplicado exacto).
- **`id_del_proceso`:** un registro por proceso.
- **`id_del_proceso` + `id_adjudicacion`:** un registro por adjudicación dentro de un proceso.
- **`id_adjudicacion`:** identificador de la adjudicación por sí solo.

Se mide sobre todo el bronze y sobre los registros con `adjudicado = True`, que son los que pasarán a gold.

In [2]:
# T1.1 · Bronze de entrada
bronze = spark.read.parquet(BRONZE_PARQUET_UNPARTITIONED)

# fecha_ingesta_utc la pone el proceso de carga; no describe el contrato
columnas_contenido = [c for c in bronze.columns if c != "fecha_ingesta_utc"]

filas_bronze = bronze.count()
filas_adjudicadas = bronze.filter(F.col("adjudicado") == True).count()
print(f"Filas en el bronze:      {filas_bronze:,}")
print(f"Filas con adjudicado=True: {filas_adjudicadas:,}")
print(f"Columnas de contenido:   {len(columnas_contenido)}")

Filas en el bronze:      9,099,432
Filas con adjudicado=True: 872,756
Columnas de contenido:   59


### Tabla de diagnóstico por clave candidata

*Filas sobrantes* = filas − claves distintas: son las filas que desaparecerían si se dejara una sola por clave.

In [3]:
# T1.1 · Tabla de diagnóstico: filas, claves distintas y filas sobrantes por clave candidata
from functools import reduce
from operator import or_
import pandas as pd

claves_candidatas = {
    "fila completa (duplicado exacto)": columnas_contenido,
    "id_del_proceso": ["id_del_proceso"],
    "id_del_proceso + id_adjudicacion": ["id_del_proceso", "id_adjudicacion"],
    "id_adjudicacion": ["id_adjudicacion"],
}

def diagnosticar(df, universo):
    filas = df.count()
    resultado = []
    for nombre, columnas in claves_candidatas.items():
        if len(columnas) > 2:
            # Fila completa: se compara un hash de 64 bits de todas las columnas
            # (con 9 millones de filas la probabilidad de colisión es despreciable)
            distintas = df.select(F.xxhash64(*columnas).alias("h")).distinct().count()
            nulas = "n/a"
        else:
            distintas = df.select(*columnas).distinct().count()
            nulas = df.filter(reduce(or_, [F.col(c).isNull() for c in columnas])).count()
        resultado.append({
            "universo": universo,
            "clave candidata": nombre,
            "filas": filas,
            "claves distintas": distintas,
            "filas sobrantes": filas - distintas,
            "% sobrante": round(100 * (filas - distintas) / filas, 2) if filas else 0.0,
            "filas con clave nula": nulas,
        })
    return resultado

diagnostico_claves = pd.DataFrame(
    diagnosticar(bronze, "todo el bronze")
    + diagnosticar(bronze.filter(F.col("adjudicado") == True), "solo adjudicados")
)
display(diagnostico_claves)

,universo,clave candidata,filas,claves distintas,filas sobrantes,% sobrante,filas con clave nula
0,todo el bronze,fila completa (duplicado exacto),9099432,8963967,135465,1.49,n/a
1,todo el bronze,id_del_proceso,9099432,8667332,432100,4.75,0
2,todo el bronze,id_del_proceso + id_adjudicacion,9099432,8694126,405306,4.45,8226676
3,todo el bronze,id_adjudicacion,9099432,547419,8552013,93.98,8226676
4,solo adjudicados,fila completa (duplicado exacto),872756,817259,55497,6.36,n/a
5,solo adjudicados,id_del_proceso,872756,520624,352132,40.35,0
6,solo adjudicados,id_del_proceso + id_adjudicacion,872756,547418,325338,37.28,0
7,solo adjudicados,id_adjudicacion,872756,547418,325338,37.28,0


### Distribución de repeticiones por proceso

In [4]:
# T1.1 · ¿Cuántas veces se repite cada id_del_proceso?
repeticiones_proceso = (
    bronze.groupBy("id_del_proceso").count()
    .withColumn("veces", F.least(F.col("count"), F.lit(10)))
    .groupBy("veces")
    .agg(F.count("*").alias("procesos"), F.sum("count").alias("filas"))
    .orderBy("veces")
    .collect()
)
repeticiones_proceso = pd.DataFrame([r.asDict() for r in repeticiones_proceso])
repeticiones_proceso["veces"] = repeticiones_proceso["veces"].map(lambda v: "10 o más" if v == 10 else str(v))
display(repeticiones_proceso)

,veces,procesos,filas
0,1,8583881,8583881
1,2,50766,101532
2,3,10958,32874
3,4,8696,34784
4,5,2734,13670
5,6,2271,13626
6,7,761,5327
7,8,1041,8328
8,9,1310,11790
9,10 o más,4914,293620


### Qué cambia entre las filas de un mismo proceso

Si en un proceso repetido cambia `id_adjudicacion`, las filas son adjudicaciones distintas (lotes o varios contratistas). Si solo cambian fechas de publicación o el estado, son versiones sucesivas del mismo proceso.

In [5]:
# T1.1 · Dentro de un mismo id_del_proceso repetido, ¿qué columnas cambian?
# Para cada proceso repetido se cuenta cuántos valores distintos tiene cada columna
# (se usa un hash por valor para no cargar textos largos en memoria; el nulo cuenta como valor).
procesos_repetidos = (
    bronze.groupBy("id_del_proceso").count()
    .filter(F.col("count") > 1)
    .select("id_del_proceso")
)
columnas_a_revisar = [c for c in columnas_contenido if c != "id_del_proceso"]

variacion = (
    bronze.join(procesos_repetidos, "id_del_proceso", "left_semi")
    .groupBy("id_del_proceso")
    .agg(*[F.size(F.collect_set(F.xxhash64(F.col(c)))).alias(c) for c in columnas_a_revisar])
)
total_repetidos = procesos_repetidos.count()

fila = variacion.agg(*[F.sum(F.when(F.col(c) > 1, 1).otherwise(0)).alias(c) for c in columnas_a_revisar]).first()
columnas_que_cambian = pd.DataFrame(
    [(c, fila[c] or 0, round(100 * (fila[c] or 0) / total_repetidos, 2) if total_repetidos else 0.0)
     for c in columnas_a_revisar],
    columns=["columna", "procesos repetidos donde cambia", "% de procesos repetidos"],
).sort_values("procesos repetidos donde cambia", ascending=False)

print(f"Procesos con más de una fila: {total_repetidos:,}")
display(columnas_que_cambian[columnas_que_cambian["procesos repetidos donde cambia"] > 0].reset_index(drop=True))

Procesos con más de una fila: 83,451


,columna,procesos repetidos donde cambia,% de procesos repetidos
0,valor_total_adjudicacion,22389,26.83
1,id_adjudicacion,18308,21.94
2,nombre_proveedor_adjudicado,14393,17.25
3,codigo_proveedor,14393,17.25
4,fecha_adjudicacion,14099,16.89
5,nit_proveedor_adjudicado,10433,12.50
6,departamento_proveedor,9889,11.85
7,ciudad_proveedor,9541,11.43
8,nombre_adjudicador,2582,3.09


### Muestra de procesos repetidos para revisión manual

In [6]:
# T1.1 · Muestra reproducible de procesos repetidos para revisarlos a mano
# Se eligen los 5 procesos repetidos con el hash más bajo: siempre salen los mismos.
columnas_muestra = [
    "id_del_proceso", "referencia_del_proceso", "fase", "estado_procedimiento",
    "adjudicado", "id_adjudicacion", "numero_lotes",
    "fecha_publicacion_proceso", "fecha_ultima_publicacion", "fecha_adjudicacion",
    "precio_base", "valor_total_adjudicacion",
    "nit_proveedor_adjudicado", "nombre_proveedor_adjudicado",
]
ids_muestra = [
    r.id_del_proceso
    for r in procesos_repetidos.orderBy(F.xxhash64("id_del_proceso"), "id_del_proceso").limit(5).collect()
]
muestra_repetidos = (
    bronze.filter(F.col("id_del_proceso").isin(ids_muestra))
    .select(*columnas_muestra)
    .orderBy("id_del_proceso", "fecha_ultima_publicacion", "id_adjudicacion")
    .collect()
)
muestra_repetidos = pd.DataFrame([r.asDict() for r in muestra_repetidos], columns=columnas_muestra)
pd.set_option("display.max_columns", None)
display(muestra_repetidos)

,id_del_proceso,referencia_del_proceso,fase,estado_procedimiento,adjudicado,id_adjudicacion,numero_lotes,fecha_publicacion_proceso,fecha_ultima_publicacion,fecha_adjudicacion,precio_base,valor_total_adjudicacion,nit_proveedor_adjudicado,nombre_proveedor_adjudicado
0,CO1.REQ.407030,PN DIBIE SA 023 2018 (Presentación de oferta),Presentación de oferta,Seleccionado,True,CO1.AWD.300105,0,2018-04-10,2018-04-10,NaT,103805319.00,101533300.00,None,VICARTECHZ SAS
1,CO1.REQ.407030,PN DIBIE SA 023 2018 (Presentación de oferta),Presentación de oferta,Seleccionado,True,CO1.AWD.301134,0,2018-04-10,2018-04-10,2018-06-08,103805319.00,95532681.00,None,TEAM CORP SAS
2,CO1.REQ.5912382,CTO 091 MT 2024,Presentación de oferta,Seleccionado,False,None,0,2024-03-08,2024-03-08,NaT,50400000.00,0.00,None,None
3,CO1.REQ.5912382,CTO 091 MT 2024,Presentación de oferta,Seleccionado,False,None,0,2024-03-08,2024-03-08,NaT,50400000.00,0.00,None,None
4,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
5,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
6,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
7,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
8,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
9,CO1.REQ.6296380,413-MC-026-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1941961,0,2024-05-27,2024-05-27,NaT,5130000.00,4288811.00,None,SERHE H2O


In [7]:
# T1.1 · Dentro de una misma adjudicación, ¿qué cambia cuando las filas NO son duplicados exactos?
adjudicados_sin_exactos = (
    bronze.filter(F.col("adjudicado") == True)
    .dropDuplicates(columnas_contenido)
)
adjudicaciones_repetidas = (
    adjudicados_sin_exactos.groupBy("id_adjudicacion").count()
    .filter(F.col("count") > 1)
    .select("id_adjudicacion")
)
filas_adj_sin_exactos = adjudicados_sin_exactos.count()
total_adj_repetidas = adjudicaciones_repetidas.count()

columnas_adj = [c for c in columnas_contenido if c != "id_adjudicacion"]
fila_adj = (
    adjudicados_sin_exactos.join(adjudicaciones_repetidas, "id_adjudicacion", "left_semi")
    .groupBy("id_adjudicacion")
    .agg(*[F.size(F.collect_set(F.xxhash64(F.col(c)))).alias(c) for c in columnas_adj])
    .agg(*[F.sum(F.when(F.col(c) > 1, 1).otherwise(0)).alias(c) for c in columnas_adj])
    .first()
)
cambios_adjudicacion = pd.DataFrame(
    [(c, fila_adj[c] or 0, round(100 * (fila_adj[c] or 0) / total_adj_repetidas, 2) if total_adj_repetidas else 0.0)
     for c in columnas_adj],
    columns=["columna", "adjudicaciones donde cambia", "% de adjudicaciones repetidas"],
).sort_values("adjudicaciones donde cambia", ascending=False)

print(f"Filas adjudicadas sin duplicados exactos: {filas_adj_sin_exactos:,}")
print(f"Adjudicaciones distintas:                 {adjudicados_sin_exactos.select('id_adjudicacion').distinct().count():,}")
print(f"Adjudicaciones con más de una versión:    {total_adj_repetidas:,}")
display(cambios_adjudicacion[cambios_adjudicacion["adjudicaciones donde cambia"] > 0].reset_index(drop=True))

Filas adjudicadas sin duplicados exactos: 817,259
Adjudicaciones distintas:                 547,418
Adjudicaciones con más de una versión:    16,616


,columna,adjudicaciones donde cambia,% de adjudicaciones repetidas
0,valor_total_adjudicacion,16110,96.95
1,codigo_proveedor,10571,63.62
2,nombre_proveedor_adjudicado,10571,63.62
3,nit_proveedor_adjudicado,7893,47.50
4,departamento_proveedor,7333,44.13
5,ciudad_proveedor,7094,42.69
6,fecha_adjudicacion,163,0.98


In [8]:
# T1.1 · Muestra reproducible de adjudicaciones con varias versiones distintas
columnas_que_varian = cambios_adjudicacion.loc[
    cambios_adjudicacion["adjudicaciones donde cambia"] > 0, "columna"
].tolist()[:10]
ids_adj_muestra = [
    r.id_adjudicacion
    for r in adjudicaciones_repetidas.orderBy(F.xxhash64("id_adjudicacion"), "id_adjudicacion").limit(5).collect()
]
muestra_adjudicaciones = (
    adjudicados_sin_exactos.filter(F.col("id_adjudicacion").isin(ids_adj_muestra))
    .select("id_adjudicacion", "id_del_proceso", *columnas_que_varian)
    .orderBy("id_adjudicacion", *columnas_que_varian)
    .collect()
)
display(pd.DataFrame([r.asDict() for r in muestra_adjudicaciones],
                     columns=["id_adjudicacion", "id_del_proceso", *columnas_que_varian]))

,id_adjudicacion,id_del_proceso,valor_total_adjudicacion,codigo_proveedor,nombre_proveedor_adjudicado,nit_proveedor_adjudicado,departamento_proveedor,ciudad_proveedor,fecha_adjudicacion
0,CO1.AWD.1346458,CO1.REQ.3045146,50000000.00,703956128,ESTUDIOS OFTALMOLOGICOS S.A.S.,None,None,None,2022-06-22
1,CO1.AWD.1346458,CO1.REQ.3045146,55000000.00,703956128,ESTUDIOS OFTALMOLOGICOS S.A.S.,None,None,None,2022-06-22
2,CO1.AWD.1346458,CO1.REQ.3045146,75000000.00,703956128,ESTUDIOS OFTALMOLOGICOS S.A.S.,None,None,None,2022-06-22
3,CO1.AWD.1492664,CO1.REQ.3662832,106200000.00,705772788,JEILER ANDRES GALVEZ LONDOÑO,None,None,None,2022-12-13
4,CO1.AWD.1492664,CO1.REQ.3662832,213800000.00,705772788,JEILER ANDRES GALVEZ LONDOÑO,None,None,None,2022-12-13
5,CO1.AWD.1574325,CO1.REQ.4257936,90000000.00,701563397,JAVIER OSVALDO RAMIREZ MARTINEZ,None,None,None,2023-03-29
6,CO1.AWD.1574325,CO1.REQ.4257936,90000000.00,703517524,M3 SOLUCIONES INTEGRALES SAS,900413586,Atlántico,Barranquilla,2023-03-29
7,CO1.AWD.1574325,CO1.REQ.4257936,240000000.00,701563397,JAVIER OSVALDO RAMIREZ MARTINEZ,None,None,None,2023-03-29
8,CO1.AWD.1574325,CO1.REQ.4257936,240000000.00,703517524,M3 SOLUCIONES INTEGRALES SAS,900413586,Atlántico,Barranquilla,2023-03-29
9,CO1.AWD.2315486,CO1.REQ.8427888,1606596857.00,732170105,CONSORCIO VIRREY ARC,901973576,None,Funza,2025-07-28


### Decisión sobre la clave natural

In [9]:
# T1.1 · Decisión sobre la clave natural (las cifras salen de las celdas anteriores)
def _fila(universo, clave):
    return diagnostico_claves[
        (diagnostico_claves["universo"] == universo) & (diagnostico_claves["clave candidata"] == clave)
    ].iloc[0]

clave_adj = _fila("solo adjudicados", "id_del_proceso + id_adjudicacion")
solo_awd = _fila("solo adjudicados", "id_adjudicacion")
exactos_adj = filas_adjudicadas - filas_adj_sin_exactos
sobrantes_conflicto = filas_adj_sin_exactos - int(clave_adj["claves distintas"])
cambian = cambios_adjudicacion[cambios_adjudicacion["adjudicaciones donde cambia"] > 0]
lista_cambian = ", ".join(f"`{c}` ({n:,})" for c, n in zip(cambian["columna"], cambian["adjudicaciones donde cambia"]))

display(Markdown(f"""
**Clave elegida: `id_del_proceso` + `id_adjudicacion`.** Un contrato adjudicado es una adjudicación (`CO1.AWD…`) dentro de un proceso (`CO1.REQ…`).

Por qué:
- `id_adjudicacion` solo tiene valor cuando `adjudicado = True` y nunca es nula en esas {filas_adjudicadas:,} filas.
- Hay **{int(clave_adj['claves distintas']):,}** parejas proceso + adjudicación distintas y **{int(solo_awd['claves distintas']):,}** adjudicaciones distintas: cada adjudicación pertenece a un único proceso.
- `id_del_proceso` solo no sirve: un mismo proceso puede tener varias adjudicaciones (lotes o varios contratistas), y deduplicar por proceso dejaría **{int(_fila('solo adjudicados', 'id_del_proceso')['claves distintas']):,}** filas, perdiendo adjudicaciones reales.
- `fecha_ultima_publicacion` no cambia dentro de un proceso, así que no sirve para elegir una "versión más reciente".

Por qué se repiten las {filas_adjudicadas - int(clave_adj['claves distintas']):,} filas sobrantes entre los adjudicados:
1. **Duplicados exactos:** {exactos_adj:,} filas idénticas en todas las columnas (por ejemplo, una misma adjudicación repetida 5 veces).
2. **Combinaciones de valor y proveedor:** {sobrantes_conflicto:,} filas en {total_adj_repetidas:,} adjudicaciones ({100 * total_adj_repetidas / int(clave_adj['claves distintas']):.2f} % del total) que repiten la adjudicación con otro valor u otro proveedor. Las columnas que cambian son: {lista_cambian}. El resto de columnas (entidad, proceso, modalidad, fechas de publicación) es idéntico. Las filas son el cruce de varios valores con varios proveedores (2 valores × 2 proveedores = 4 filas) y el dato abierto no indica qué valor corresponde a qué proveedor, así que **no se pueden sumar**: inflarían el valor contratado.

Regla para T1.2 (una fila por clave, con `row_number` y este orden):
1. `fecha_adjudicacion` no nula primero.
2. Mayor `valor_total_adjudicacion`.
3. Proveedor identificado primero: con `nit_proveedor_adjudicado` y luego con `codigo_proveedor`.
4. Desempate fijo: `codigo_proveedor`, `nit_proveedor_adjudicado` y `nombre_proveedor_adjudicado` en orden ascendente.

Las filas descartadas se guardan aparte con su motivo (`duplicado_exacto` o `version_alternativa`). Para los registros no adjudicados (`id_adjudicacion` nula) la clave queda en una fila por proceso.
"""))


**Clave elegida: `id_del_proceso` + `id_adjudicacion`.** Un contrato adjudicado es una adjudicación (`CO1.AWD…`) dentro de un proceso (`CO1.REQ…`).

Por qué:
- `id_adjudicacion` solo tiene valor cuando `adjudicado = True` y nunca es nula en esas 872,756 filas.
- Hay **547,418** parejas proceso + adjudicación distintas y **547,418** adjudicaciones distintas: cada adjudicación pertenece a un único proceso.
- `id_del_proceso` solo no sirve: un mismo proceso puede tener varias adjudicaciones (lotes o varios contratistas), y deduplicar por proceso dejaría **520,624** filas, perdiendo adjudicaciones reales.
- `fecha_ultima_publicacion` no cambia dentro de un proceso, así que no sirve para elegir una "versión más reciente".

Por qué se repiten las 325,338 filas sobrantes entre los adjudicados:
1. **Duplicados exactos:** 55,497 filas idénticas en todas las columnas (por ejemplo, una misma adjudicación repetida 5 veces).
2. **Combinaciones de valor y proveedor:** 269,841 filas en 16,616 adjudicaciones (3.04 % del total) que repiten la adjudicación con otro valor u otro proveedor. Las columnas que cambian son: `valor_total_adjudicacion` (16,110), `codigo_proveedor` (10,571), `nombre_proveedor_adjudicado` (10,571), `nit_proveedor_adjudicado` (7,893), `departamento_proveedor` (7,333), `ciudad_proveedor` (7,094), `fecha_adjudicacion` (163). El resto de columnas (entidad, proceso, modalidad, fechas de publicación) es idéntico. Las filas son el cruce de varios valores con varios proveedores (2 valores × 2 proveedores = 4 filas) y el dato abierto no indica qué valor corresponde a qué proveedor, así que **no se pueden sumar**: inflarían el valor contratado.

Regla para T1.2 (una fila por clave, con `row_number` y este orden):
1. `fecha_adjudicacion` no nula primero.
2. Mayor `valor_total_adjudicacion`.
3. Proveedor identificado primero: con `nit_proveedor_adjudicado` y luego con `codigo_proveedor`.
4. Desempate fijo: `codigo_proveedor`, `nit_proveedor_adjudicado` y `nombre_proveedor_adjudicado` en orden ascendente.

Las filas descartadas se guardan aparte con su motivo (`duplicado_exacto` o `version_alternativa`). Para los registros no adjudicados (`id_adjudicacion` nula) la clave queda en una fila por proceso.


## T1.2 Deduplicación determinista (capa silver)

Se deja una sola fila por clave (`id_del_proceso`, `id_adjudicacion`), con la regla aprobada en T1.1, usando `row_number` sobre una ventana con orden explícito. Las filas que no se conservan van a una tabla aparte con su motivo:
- `duplicado_exacto`: idéntica en todas las columnas a otra fila de la misma clave.
- `version_alternativa`: misma clave con otro valor, proveedor o fecha.

Salidas: `silver/secop/contratos` y `silver/secop/descartados`.

In [3]:
# T1.2 · Deduplicación determinista: una fila por (id_del_proceso, id_adjudicacion)
import pandas as pd

CLAVE_SILVER = ["id_del_proceso", "id_adjudicacion"]

# Orden de preferencia acordado en T1.1 (la primera fila de cada clave es la que se conserva)
ORDEN_SILVER = [
    F.col("fecha_adjudicacion").isNull().asc(),          # 1. con fecha de adjudicación primero
    F.col("valor_total_adjudicacion").desc_nulls_last(),  # 2. mayor valor adjudicado
    F.col("nit_proveedor_adjudicado").isNull().asc(),     # 3. proveedor con NIT...
    F.col("codigo_proveedor").isNull().asc(),             #    ...y luego con código
    F.col("codigo_proveedor").asc_nulls_last(),           # 4. desempate fijo
    F.col("nit_proveedor_adjudicado").asc_nulls_last(),
    F.col("nombre_proveedor_adjudicado").asc_nulls_last(),
]

def deduplicar(df):
    """Devuelve el DataFrame con 'rn' (1 = se conserva) y 'motivo_descarte'."""
    huella = F.xxhash64(*columnas_contenido)  # identifica filas idénticas en todas las columnas
    por_clave = Window.partitionBy(*CLAVE_SILVER).orderBy(*ORDEN_SILVER)
    por_clave_y_huella = Window.partitionBy(*CLAVE_SILVER, "_huella").orderBy(*ORDEN_SILVER)
    return (
        df.withColumn("_huella", huella)
        .withColumn("rn", F.row_number().over(por_clave))
        .withColumn("_rn_huella", F.row_number().over(por_clave_y_huella))
        .withColumn(
            "motivo_descarte",
            F.when(F.col("rn") == 1, F.lit(None).cast("string"))
             .when(F.col("_rn_huella") > 1, F.lit("duplicado_exacto"))
             .otherwise(F.lit("version_alternativa")),
        )
        .drop("_huella", "_rn_huella")
    )

def resumen(df):
    """Conteo, sumas de valores y suma de huellas: si dos ejecuciones dan lo mismo, el resultado es idéntico."""
    return df.agg(
        F.count("*").alias("filas"),
        F.sum("valor_total_adjudicacion").alias("suma_valor_adjudicado"),
        F.sum("precio_base").alias("suma_precio_base"),
        F.sum(F.xxhash64(*columnas_contenido).cast("decimal(38,0)")).alias("suma_huellas"),
    ).first().asDict()

### Escritura de silver

`RECONSTRUIR_SILVER = True` recalcula y escribe silver y los descartados; con `False` se reutiliza lo que ya está en MinIO.

In [4]:
# T1.2 · Escritura de silver y de las filas descartadas
# True la primera vez (escribe silver y descartados); False reutiliza lo que ya está en MinIO.
# Silver se escribió el 13-nov-2026 con True; la salida de abajo es la de esa ejecución.
RECONSTRUIR_SILVER = False

if RECONSTRUIR_SILVER:
    marcado = deduplicar(bronze).persist(StorageLevel.DISK_ONLY)
    (
        marcado.filter(F.col("rn") == 1)
        .drop("rn", "motivo_descarte")
        .write.mode(WRITE_MODE).option("compression", "snappy")
        .parquet(SILVER_PARQUET)
    )
    (
        marcado.filter(F.col("rn") > 1)
        .drop("rn")
        .write.mode(WRITE_MODE).option("compression", "snappy")
        .parquet(SILVER_DESCARTADOS_PARQUET)
    )
    marcado.unpersist()

silver = spark.read.parquet(SILVER_PARQUET)
descartados = spark.read.parquet(SILVER_DESCARTADOS_PARQUET)

### Criterios de aceptación

1. Silver tiene una fila por clave (se comprueba con `assert`).
2. Silver + descartadas = bronze: ninguna fila se pierde sin explicación.
3. Determinismo: se repite la deduplicación y se compara con lo escrito el conteo, la suma de `valor_total_adjudicacion`, la de `precio_base` y la suma de las huellas (`xxhash64`) de todas las filas. Si coinciden, las dos ejecuciones conservaron exactamente las mismas filas.

In [5]:
# T1.2 · Criterios de aceptación
filas_silver = silver.count()
filas_descartadas = descartados.count()
claves_distintas_bronze = bronze.select(*CLAVE_SILVER).distinct().count()

print(f"Bronze:                    {filas_bronze:,}")
print(f"Silver:                    {filas_silver:,}")
print(f"Descartadas:               {filas_descartadas:,}")
print(f"Claves distintas (bronze): {claves_distintas_bronze:,}")

# (1) Una fila por clave
assert filas_silver == claves_distintas_bronze, "Silver no tiene una fila por clave"
assert silver.groupBy(*CLAVE_SILVER).count().filter(F.col("count") > 1).count() == 0, "Hay claves repetidas en silver"
# Ninguna fila se pierde: silver + descartadas = bronze
assert filas_silver + filas_descartadas == filas_bronze, "Silver + descartadas no suma el bronze"

display(pd.DataFrame([r.asDict() for r in descartados.groupBy("motivo_descarte").count().orderBy("motivo_descarte").collect()]))

# (2) Determinismo: se repite la deduplicación en memoria y se compara con lo escrito
resumen_escrito = resumen(silver)
resumen_repetido = resumen(deduplicar(bronze).filter(F.col("rn") == 1).drop("rn", "motivo_descarte"))
display(pd.DataFrame([resumen_escrito, resumen_repetido], index=["silver escrito", "segunda ejecución"]))
assert resumen_escrito == resumen_repetido, "Dos ejecuciones dieron resultados distintos"
print("Criterios de aceptación de T1.2: OK")

Bronze:                    9,099,432
Silver:                    8,694,126
Descartadas:               405,306
Claves distintas (bronze): 8,694,126


,motivo_descarte,count
0,duplicado_exacto,135465
1,version_alternativa,269841


,filas,suma_valor_adjudicado,suma_precio_base,suma_huellas
silver escrito,8694126,570571474774174068.00,194476622903948512.00,-5729303209801678712062
segunda ejecución,8694126,570571474774174068.00,194476622903948512.00,-5729303209801678712062


Criterios de aceptación de T1.2: OK


## T1.3 Tabla gold de adjudicaciones

Gold parte de silver y conserva solo los contratos adjudicados (`adjudicado = True`) con fecha de adjudicación. Tiene una fila por adjudicación, con la entidad, el contratista, el valor y la fecha, más `anio_adjudicacion` y `anio_mes` para las agregaciones del Corte 3. Los identificadores siguen la regla del Punto 3 del Corte 2: la entidad por NIT y, si falta, por nombre; el contratista por NIT, luego por código y luego por nombre.

Un adjudicado no pasa a gold si le falta la fecha de adjudicación, si esa fecha es posterior a la descarga de los datos (2 de septiembre de 2026), si no se puede identificar la entidad o el contratista, o si el valor falta o es negativo. Esos registros se guardan en `gold/secop/excluidos` con su motivo.

Salida: `gold/secop/adjudicaciones`, en Parquet particionado por `anio_adjudicacion`.

In [5]:
# T1.3 · Construcción de gold: un registro por contrato adjudicado
FECHA_CORTE_DATOS = "2026-09-02"  # descarga del CSV de SECOP II; una adjudicación posterior es una fecha futura

def texto_normalizado(columna):
    """Mayúsculas, espacios simples y sin marcadores de ausencia (misma regla que el Punto 3 del Corte 2)."""
    valor = F.upper(F.trim(F.regexp_replace(F.col(columna).cast("string"), r"\s+", " ")))
    return F.when(~valor.isin("", "NO DEFINIDO", "NO DEFINIDA", "NO ADJUDICADO", "N/A", "NA", "NULL", "0", "-1"), valor)

# Identificadores: entidad por NIT y luego nombre; contratista por NIT, luego código y luego nombre (Punto 3 del Corte 2)
id_entidad = F.coalesce(
    F.concat(F.lit("NIT:"), texto_normalizado("nit_entidad")),
    F.concat(F.lit("NOMBRE:"), texto_normalizado("entidad")),
)
id_contratista = F.coalesce(
    F.concat(F.lit("NIT:"), texto_normalizado("nit_proveedor_adjudicado")),
    F.concat(F.lit("CODIGO:"), texto_normalizado("codigo_proveedor")),
    F.concat(F.lit("NOMBRE:"), texto_normalizado("nombre_proveedor_adjudicado")),
)

adjudicados_silver = silver.filter(F.col("adjudicado") == True)

candidatos_gold = adjudicados_silver.select(
    "id_del_proceso",
    "id_adjudicacion",
    id_entidad.alias("id_entidad"),
    F.col("nit_entidad"),
    F.col("entidad").alias("nombre_entidad"),
    "departamento_entidad",
    "ciudad_entidad",
    id_contratista.alias("id_contratista"),
    F.col("nombre_proveedor_adjudicado").alias("nombre_contratista"),
    F.col("departamento_proveedor").alias("departamento_contratista"),
    "modalidad_contratacion",
    "tipo_contrato",
    "valor_total_adjudicacion",
    F.to_date("fecha_adjudicacion").alias("fecha_adjudicacion"),
).withColumn(
    "motivo_exclusion",
    F.when(F.col("fecha_adjudicacion").isNull(), F.lit("sin_fecha_adjudicacion"))
     .when(F.col("fecha_adjudicacion") > F.lit(FECHA_CORTE_DATOS).cast("date"), F.lit("fecha_futura"))
     .when(F.col("id_entidad").isNull(), F.lit("sin_entidad"))
     .when(F.col("id_contratista").isNull(), F.lit("sin_contratista"))
     .when(F.col("valor_total_adjudicacion").isNull(), F.lit("sin_valor"))
     .when(F.col("valor_total_adjudicacion") < 0, F.lit("valor_negativo")),
)

gold_df = (
    candidatos_gold.filter(F.col("motivo_exclusion").isNull())
    .drop("motivo_exclusion")
    .withColumn("anio_adjudicacion", F.year("fecha_adjudicacion"))
    .withColumn("anio_mes", F.date_format("fecha_adjudicacion", "yyyy-MM"))
)
excluidos_gold = candidatos_gold.filter(F.col("motivo_exclusion").isNotNull())

### Escritura de gold

In [6]:
# T1.3 · Escritura de gold (particionado por año de adjudicación) y de los excluidos
# True la primera vez (escribe gold y excluidos); False reutiliza lo que ya está en MinIO.
# Gold se escribió el 13-nov-2026 con True; la salida de abajo es la de esa ejecución.
RECONSTRUIR_GOLD = False

if RECONSTRUIR_GOLD:
    (
        gold_df
        .repartition("anio_adjudicacion")
        .write.mode(WRITE_MODE).option("compression", "snappy")
        .partitionBy("anio_adjudicacion")
        .parquet(GOLD_PARQUET)
    )
    excluidos_gold.write.mode(WRITE_MODE).option("compression", "snappy").parquet(GOLD_EXCLUIDOS_PARQUET)

gold = spark.read.parquet(GOLD_PARQUET)
excluidos = spark.read.parquet(GOLD_EXCLUIDOS_PARQUET)

### Esquema de gold

In [7]:
# T1.3 · Esquema documentado (tipo y significado de cada columna)
significado_gold = {
    "id_del_proceso": "Identificador del proceso de contratación en SECOP II (CO1.REQ…).",
    "id_adjudicacion": "Identificador de la adjudicación (CO1.AWD…). Con id_del_proceso forma la clave de gold.",
    "id_entidad": "Identificador de la entidad: 'NIT:<nit>' o, si falta, 'NOMBRE:<nombre normalizado>'.",
    "nit_entidad": "NIT de la entidad contratante, tal como viene del bronze.",
    "nombre_entidad": "Nombre de la entidad contratante.",
    "departamento_entidad": "Departamento de la entidad (filtro por región del dashboard).",
    "ciudad_entidad": "Ciudad de la entidad.",
    "id_contratista": "Identificador del contratista: 'NIT:', luego 'CODIGO:' y luego 'NOMBRE:' (regla del Punto 3 del Corte 2).",
    "nombre_contratista": "Nombre del proveedor adjudicado.",
    "departamento_contratista": "Departamento del proveedor adjudicado (puede ser nulo).",
    "modalidad_contratacion": "Modalidad de contratación del proceso.",
    "tipo_contrato": "Tipo de contrato.",
    "valor_total_adjudicacion": "Valor adjudicado en pesos colombianos (base de todos los montos del Corte 3).",
    "fecha_adjudicacion": "Fecha de adjudicación (sin hora).",
    "anio_mes": "Año y mes de adjudicación, 'AAAA-MM' (ventanas mensuales y gráficas).",
    "anio_adjudicacion": "Año de adjudicación; columna de partición de gold.",
}
esquema_gold = pd.DataFrame(
    [(f.name, f.dataType.simpleString(), significado_gold.get(f.name, "")) for f in gold.schema.fields],
    columns=["columna", "tipo", "significado"],
)
assert (esquema_gold["significado"] != "").all(), "Hay columnas sin documentar"
display(esquema_gold)

,columna,tipo,significado
0,id_del_proceso,string,Identificador del proceso de contratación en S...
1,id_adjudicacion,string,Identificador de la adjudicación (CO1.AWD…). C...
2,id_entidad,string,"Identificador de la entidad: 'NIT:<nit>' o, si..."
3,nit_entidad,bigint,"NIT de la entidad contratante, tal como viene ..."
4,nombre_entidad,string,Nombre de la entidad contratante.
5,departamento_entidad,string,Departamento de la entidad (filtro por región ...
6,ciudad_entidad,string,Ciudad de la entidad.
7,id_contratista,string,"Identificador del contratista: 'NIT:', luego '..."
8,nombre_contratista,string,Nombre del proveedor adjudicado.
9,departamento_contratista,string,Departamento del proveedor adjudicado (puede s...


### Criterios de aceptación

1. El esquema está documentado con el tipo y el significado de cada columna (celda anterior).
2. No hay valores adjudicados negativos ni fechas anteriores a 1900 o posteriores al corte de los datos (se comprueba con `assert`).
3. Los registros excluidos están contados por motivo, y gold + excluidos = adjudicados de silver.

In [8]:
# T1.3 · Criterios de aceptación
filas_gold = gold.count()
filas_excluidas = excluidos.count()
filas_adjudicadas_silver = adjudicados_silver.count()

print(f"Adjudicados en silver:  {filas_adjudicadas_silver:,}")
print(f"Gold:                   {filas_gold:,}")
print(f"Excluidos:              {filas_excluidas:,}")
assert filas_gold + filas_excluidas == filas_adjudicadas_silver, "Gold + excluidos no suma los adjudicados de silver"
assert gold.select("id_del_proceso", "id_adjudicacion").distinct().count() == filas_gold, "Clave repetida en gold"

# (3) Registros excluidos, por motivo
display(pd.DataFrame([r.asDict() for r in excluidos.groupBy("motivo_exclusion").count().orderBy(F.desc("count")).collect()]))

# (2) Valores y fechas fuera de rango
rangos = gold.agg(
    F.min("valor_total_adjudicacion").alias("valor_minimo"),
    F.max("valor_total_adjudicacion").alias("valor_maximo"),
    F.sum(F.when(F.col("valor_total_adjudicacion") == 0, 1).otherwise(0)).alias("valores_en_cero"),
    F.min("fecha_adjudicacion").alias("fecha_minima"),
    F.max("fecha_adjudicacion").alias("fecha_maxima"),
).first()
print(rangos)
assert rangos["valor_minimo"] >= 0, "Hay valores adjudicados negativos"
assert str(rangos["fecha_minima"]) >= "1900-01-01", "Hay fechas anteriores a 1900"
assert str(rangos["fecha_maxima"]) <= FECHA_CORTE_DATOS, "Hay fechas posteriores al corte de los datos"

# Contratos por año, para ver la cobertura temporal de gold
display(pd.DataFrame([r.asDict() for r in gold.groupBy("anio_adjudicacion")
                      .agg(F.count("*").alias("contratos"), F.sum("valor_total_adjudicacion").alias("valor_total"))
                      .orderBy("anio_adjudicacion").collect()]))
print("Criterios de aceptación de T1.3: OK")

Adjudicados en silver:  547,418
Gold:                   530,192
Excluidos:              17,226


,motivo_exclusion,count
0,sin_fecha_adjudicacion,17053
1,sin_contratista,173


Row(valor_minimo=Decimal('0.00'), valor_maximo=Decimal('89069800047730398.00'), valores_en_cero=1527, fecha_minima=datetime.date(2015, 6, 11), fecha_maxima=datetime.date(2026, 9, 1))


,anio_adjudicacion,contratos,valor_total
0,2015,29,9144206616.00
1,2016,1859,70758269283056.00
2,2017,18284,67600174083503.00
3,2018,32141,13927437992466482.00
4,2019,33193,8857511584117998.00
5,2020,39160,6999931462533473.00
6,2021,56100,13540647558190157.00
7,2022,66717,100469226243036004.00
8,2023,71544,8869840410436041.00
9,2024,74067,6620289880191104.00


Criterios de aceptación de T1.3: OK


In [8]:
# T1.3 · Observaciones para T1.5: valores atípicos en gold (no se modifican aquí)
mayores = gold.orderBy(F.desc("valor_total_adjudicacion")).select(
    "id_adjudicacion", "nombre_entidad", "nombre_contratista", "fecha_adjudicacion", "valor_total_adjudicacion"
).limit(10).collect()
display(pd.DataFrame([r.asDict() for r in mayores]))

umbral_billon = 10**12  # un billón de pesos
sobre_umbral = gold.filter(F.col("valor_total_adjudicacion") > umbral_billon).agg(
    F.count("*").alias("n"), F.sum("valor_total_adjudicacion").alias("suma")).first()
total_gold = gold.agg(F.sum("valor_total_adjudicacion")).first()[0]
display(Markdown(f"""
**Observaciones para T1.5 (valores atípicos).** Gold conserva los valores tal como vienen de SECOP II; la decisión de marcarlos o excluirlos es de T1.5.
- {rangos['valores_en_cero']:,} adjudicaciones tienen valor 0.
- {sobre_umbral['n']:,} adjudicaciones superan un billón de pesos y suman el {100 * sobre_umbral['suma'] / total_gold:.2f} % del valor total de gold. El máximo es {rangos['valor_maximo']:,.0f} pesos, varias veces el presupuesto anual de la Nación, así que se trata de errores de captura.
- 2026 solo llega hasta {rangos['fecha_maxima']} (fecha de descarga de los datos), así que es un año incompleto.
"""))

,id_adjudicacion,nombre_entidad,nombre_contratista,fecha_adjudicacion,valor_total_adjudicacion
0,CO1.AWD.1445345,EAG,URBE,2022-10-18,89069800047730398.00
1,CO1.AWD.1192019,INSTITUTO MUNICIPAL DEL DEPORTE Y LA RECREACIO...,MAQUINARIAS PARA AREAS VERDES SA MAQUIVER SA,2021-12-09,12858450316000000.00
2,CO1.AWD.348301,MUNICIPIO DE FIRAVITOBA,FUNDACION SOÑANDO POR NUESTRO MUNDO,2018-08-29,11877456456257536.00
3,CO1.AWD.1408949,ALCALDIA MUNICIPAL DE BUGA,RODRIGO VALENCIA VILLARREAL,2022-09-08,11015179929862500.00
4,CO1.AWD.570328,MUNICIPIO DE ENVIGADO*,CORPORACIÓN EVG EXTEMO,2019-08-21,8839189460374700.00
5,CO1.AWD.1769384,DEPARTAMENTO DE SANTANDER,CARLOS AUGUSTO HERNANDEZ SANCHEZ,2023-11-15,8811763956840000.00
6,CO1.AWD.2679736,ESE HOSPITAL LOCAL SAN JOSE DE PUEBLO VIEJO,G&C MEDICALS S.A.S,2026-08-31,8054481856630300.00
7,CO1.AWD.2253730,IMDERCOR,FUNDACIÓN PARA LA INNOVACIÓN DE LAS CIENCIAS D...,2025-05-22,6514265521000000.00
8,CO1.AWD.1840933,INSTITUCION EDUCATIVA ELISEO PAYAN,CIFRAS ASESORÍAS Y PROYECTOS CIASPRO S.A.S.,2024-02-19,6453840000000000.00
9,CO1.AWD.751206,DEPARTAMENTO DE CUNDINAMARCA- SECRETARIA DE SALUD,LABCARE DE COLOMBIA SAS,2020-05-20,6158661742599964.00



**Observaciones para T1.5 (valores atípicos).** Gold conserva los valores tal como vienen de SECOP II; la decisión de marcarlos o excluirlos es de T1.5.
- 1,527 adjudicaciones tienen valor 0.
- 48 adjudicaciones superan un billón de pesos y suman el 99.79 % del valor total de gold. El máximo es 89,069,800,047,730,398 pesos, varias veces el presupuesto anual de la Nación, así que se trata de errores de captura.
- 2026 solo llega hasta 2026-09-01 (fecha de descarga de los datos), así que es un año incompleto.


## T1.5 Reconciliación y control de calidad

Cada etapa del flujo debe sumar exactamente las filas de la anterior: lo que entra es igual a lo que sale más lo que se descarta con su motivo. La tabla sale de los datos escritos en MinIO (no de las variables de las celdas de arriba) y cada paso se comprueba con `assert`.

Después se revisan los valores atípicos de gold. Gold no se modifica: las marcas se calculan con las funciones de `src/pamd/calidad.py` y las usan las tablas resumen de T1.4, el streaming y el dashboard.

**Cómo leer la tabla.** Cada etapa es una sola operación sobre las filas que entran. Los motivos de salida son las partes en que se reparte lo que sale en *esa misma operación*, así que lo que pasa se calcula una vez por etapa: `entran − (suma de lo que sale por todos los motivos) = pasan`, sin importar cuántos motivos tenga. Por eso `entran` y `pasan` no cambian de un motivo a otro dentro de una etapa; solo cambia cuánto sale por cada uno.

Cada fila sale por **un solo** motivo: en silver, `duplicado_exacto` y `version_alternativa` son excluyentes, y en gold se asigna el primer motivo que se cumple, en este orden: `sin_fecha_adjudicacion`, `fecha_futura`, `sin_entidad`, `sin_contratista`, `sin_valor`, `valor_negativo`. Por eso una fila que incumple dos reglas se cuenta una sola vez y las salidas nunca se solapan. Los motivos con 0 filas se dejan en la tabla para dejar constancia de que se revisaron.


In [9]:
# T1.5 · Tabla de reconciliación: cada etapa suma exactamente la anterior
import pandas as pd

leidas = spark.read.parquet(STAGING_PARQUET).count()
rechazadas = spark.read.parquet(QUARANTINE_PARQUET).count()
n_bronze = spark.read.parquet(BRONZE_PARQUET_UNPARTITIONED).count()

n_silver = silver.count()
n_dup_exacto = descartados.filter(F.col("motivo_descarte") == "duplicado_exacto").count()
n_version_alt = descartados.filter(F.col("motivo_descarte") == "version_alternativa").count()
assert descartados.count() == n_dup_exacto + n_version_alt, "Hay descartes de silver con un motivo no previsto"

es_adjudicado = F.coalesce(F.col("adjudicado"), F.lit(False))
n_adjudicadas = silver.filter(es_adjudicado).count()
n_no_adjudicadas = silver.filter(~es_adjudicado).count()

n_gold = gold.count()
MOTIVOS_GOLD = ["sin_fecha_adjudicacion", "fecha_futura", "sin_entidad", "sin_contratista", "sin_valor", "valor_negativo"]
excluidos_por_motivo = {r["motivo_exclusion"]: r["count"] for r in excluidos.groupBy("motivo_exclusion").count().collect()}
assert set(excluidos_por_motivo) <= set(MOTIVOS_GOLD), f"Motivo de exclusión no previsto: {set(excluidos_por_motivo) - set(MOTIVOS_GOLD)}"

# (etapa, filas que entran, [(motivo, filas que salen)], filas que pasan a la siguiente capa)
pasos = [
    ("CSV → bronze", leidas, [("rechazada (cuarentena)", rechazadas)], n_bronze),
    ("bronze → silver", n_bronze, [("duplicado_exacto", n_dup_exacto), ("version_alternativa", n_version_alt)], n_silver),
    ("silver → adjudicadas", n_silver, [("no_adjudicada", n_no_adjudicadas)], n_adjudicadas),
    ("adjudicadas → gold", n_adjudicadas, [(m, excluidos_por_motivo.get(m, 0)) for m in MOTIVOS_GOLD], n_gold),
]

filas_tabla = []
for etapa, entran, salidas, pasan in pasos:
    assert entran - sum(n for _, n in salidas) == pasan, f"La etapa '{etapa}' no cuadra"
    filas_tabla.append({"etapa": etapa, "concepto": "entran", "filas": entran})
    for motivo, n in salidas:
        filas_tabla.append({"etapa": "", "concepto": f"− salen: {motivo}", "filas": n})
    filas_tabla.append({"etapa": "", "concepto": "= pasan a la siguiente capa", "filas": pasan})

reconciliacion = pd.DataFrame(filas_tabla)
display(
    reconciliacion.style
    .format({"filas": "{:,}"})
    .apply(lambda fila: ["font-weight: bold" if fila["concepto"].startswith("=") else "" for _ in fila], axis=1)
)

# Los extremos de la cadena: lo leído del CSV se explica completo por gold + todo lo que salió en el camino
salidas_totales = sum(n for _, _, salidas, _ in pasos for _, n in salidas)
assert leidas == n_gold + salidas_totales, "El total leído no se explica por gold + descartes"
print(f"Leídas {leidas:,} = gold {n_gold:,} + descartadas {salidas_totales:,}. No hay filas sin explicar.")

,etapa,concepto,filas
0,CSV → bronze,entran,"9,099,859"
1,,− salen: rechazada (cuarentena),427
2,,= pasan a la siguiente capa,"9,099,432"
3,bronze → silver,entran,"9,099,432"
4,,− salen: duplicado_exacto,"135,465"
5,,− salen: version_alternativa,"269,841"
6,,= pasan a la siguiente capa,"8,694,126"
7,silver → adjudicadas,entran,"8,694,126"
8,,− salen: no_adjudicada,"8,146,708"
9,,= pasan a la siguiente capa,"547,418"


Leídas 9,099,859 = gold 530,192 + descartadas 8,569,667. No hay filas sin explicar.


### Valores atípicos en gold

Primero se mira cómo se reparte el valor por orden de magnitud y qué pasaría con distintos umbrales. Esa mirada mostró que un umbral por valor no separa los errores de los contratos grandes reales; lo que sí los separa es la razón entre el valor adjudicado y el precio base (celdas siguientes).

In [10]:
# T1.5 · Distribución del valor por orden de magnitud y efecto de cada umbral posible
valor_total_gold = gold.agg(F.sum("valor_total_adjudicacion")).first()[0]

por_magnitud = (
    gold.filter(F.col("valor_total_adjudicacion") > 0)
    .withColumn("orden", F.floor(F.log10(F.col("valor_total_adjudicacion").cast("double"))).cast("int"))
    .groupBy("orden")
    .agg(F.count("*").alias("contratos"), F.sum("valor_total_adjudicacion").alias("valor"))
    .orderBy("orden")
    .collect()
)
display(pd.DataFrame([
    {"valor desde (pesos)": f"10^{r['orden']}", "contratos": r["contratos"],
     "% del valor total": 100 * float(r["valor"]) / float(valor_total_gold)}
    for r in por_magnitud
]).style.format({"contratos": "{:,}", "% del valor total": "{:.4f}"}))

umbrales = [10**10, 10**11, 10**12, 10**13]
filas_umbral = []
for u in umbrales:
    r = gold.filter(F.col("valor_total_adjudicacion") > u).agg(
        F.count("*").alias("n"), F.sum("valor_total_adjudicacion").alias("suma")).first()
    filas_umbral.append({"umbral (pesos)": f"{u:,}", "contratos por encima": r["n"],
                         "% del valor total": 100 * float(r["suma"] or 0) / float(valor_total_gold)})
display(pd.DataFrame(filas_umbral).style.format({"contratos por encima": "{:,}", "% del valor total": "{:.4f}"}))

,valor desde (pesos),contratos,% del valor total
0,10^0,676,0.0000
1,10^1,50,0.0000
2,10^2,102,0.0000
3,10^3,211,0.0000
4,10^4,488,0.0000
5,10^5,"6,135",0.0000
6,10^6,"87,052",0.0003
7,10^7,"264,253",0.0058
8,10^8,"130,066",0.0247
9,10^9,"34,455",0.0556


,umbral (pesos),contratos por encima,% del valor total
0,"10,000,000,000","5,145",99.9135
1,"100,000,000,000",409,99.8461
2,"1,000,000,000,000",48,99.7948
3,"10,000,000,000,000",21,99.7589


In [11]:
# T1.5 · Gold con marcas: razón valor adjudicado / precio base (reglas en src/pamd/calidad.py; gold no cambia)
gold_marcado = marcar_atipicos(unificar_nit_entidad(gold), silver).persist(StorageLevel.MEMORY_AND_DISK)
assert gold_marcado.count() == n_gold, "El join con silver cambió el número de contratos"

con_razon = gold_marcado.filter(F.col("razon_valor_precio_base").isNotNull())
n_con_razon = con_razon.count()
valor_con_razon = con_razon.agg(F.sum("valor_total_adjudicacion")).first()[0]

por_razon = (
    con_razon.withColumn("orden", F.floor(F.log10("razon_valor_precio_base")).cast("int"))
    .groupBy("orden")
    .agg(F.count("*").alias("contratos"), F.sum("valor_total_adjudicacion").alias("valor"))
    .orderBy("orden")
    .collect()
)
display(pd.DataFrame([
    {"razón desde": f"10^{r['orden']}", "contratos": r["contratos"],
     "% del valor": 100 * float(r["valor"]) / float(valor_con_razon)}
    for r in por_razon
]).style.format({"contratos": "{:,}", "% del valor": "{:.4f}"}))
print(f"Gold: {n_gold:,} contratos; con razón calculable: {n_con_razon:,}; sin precio base utilizable (o valor 0): {n_gold - n_con_razon:,}")

,razón desde,contratos,% del valor
0,10^-13,1,0.0000
1,10^-12,1,0.0000
2,10^-11,7,0.0000
3,10^-10,6,0.0000
4,10^-9,12,0.0000
5,10^-8,28,0.0000
6,10^-7,35,0.0000
7,10^-6,52,0.0000
8,10^-5,90,0.0000
9,10^-4,285,0.0000


Gold: 530,192 contratos; con razón calculable: 521,271; sin precio base utilizable (o valor 0): 8,921


### NIT de entidad con el dígito de verificación pegado

Un NIT de entidad de 10 dígitos puede ser en realidad un NIT de 9 dígitos con su dígito de verificación pegado (por ejemplo `NIT:8999990619` es `NIT:899999061` más el dígito 9). Eso parte una misma entidad en dos filas del ranking. `unificar_nit_entidad` los junta solo cuando el id de 9 dígitos ya existe; los demás NIT de 10 dígitos no se tocan.

In [12]:
# T1.5 · Efecto de unificar los NIT de entidad con dígito de verificación pegado
ids_antes = gold.select("id_entidad").distinct().count()
ids_despues = gold_marcado.select("id_entidad").distinct().count()
ids_gold = gold_marcado.select("id_entidad").distinct()
dv_pegado = (ids_gold.filter(F.col("id_entidad").rlike(r"^NIT:\d{10}$"))
             .withColumn("_id9", F.substring("id_entidad", 1, 13))
             .join(ids_gold.withColumnRenamed("id_entidad", "_id9"), "_id9"))
assert dv_pegado.count() == 0, "Quedan NIT con dígito de verificación pegado que duplican otra entidad"
print(f"Ids de entidad: {ids_antes:,} antes y {ids_despues:,} después de unificar ({ids_antes - ids_despues:,} unificados)")

Ids de entidad: 4,149 antes y 4,050 después de unificar (99 unificados)


### Decisión sobre los atípicos

Se **marcan**, no se borran. Gold conserva todos los contratos; las marcas las calcula `marcar_atipicos` (en `src/pamd/calidad.py`) y las tablas resumen, el flujo en streaming y el dashboard deben usar la misma función para tener una sola definición.

In [13]:
# T1.5 · Efecto de las marcas de atípicos
efecto = gold_marcado.agg(
    F.count("*").alias("contratos"),
    F.sum(F.col("es_valor_cero").cast("int")).alias("n_cero"),
    F.sum(F.col("es_valor_inconsistente").cast("int")).alias("n_inconsistente"),
    F.sum(F.col("es_valor_extremo").cast("int")).alias("n_extremo"),
    F.sum(F.col("es_atipico").cast("int")).alias("n_atipico"),
    F.sum(F.when(F.col("es_atipico"), F.col("valor_total_adjudicacion")).otherwise(0)).alias("valor_atipico"),
    F.sum("valor_total_adjudicacion").alias("valor_total"),
    F.max(F.when(~F.col("es_atipico"), F.col("valor_total_adjudicacion"))).alias("maximo_normal"),
).first().asDict()

assert efecto["contratos"] == n_gold, "las marcas no deben borrar contratos"
valor_normal = efecto["valor_total"] - efecto["valor_atipico"]
n_normales = efecto["contratos"] - efecto["n_atipico"]
prom_con = float(efecto["valor_total"]) / efecto["contratos"]
prom_sin = float(valor_normal) / n_normales

display(Markdown(f"""
**Reglas.**
- **Inconsistente:** valor adjudicado al menos {UMBRAL_RAZON_PRECIO_BASE} veces el precio base (con precio base > 1 y valor > 0). Desde esa razón la caída natural de la distribución desaparece; es el patrón de los errores de captura (típicamente ×1.000).
- **Extremo:** valor mayor que {UMBRAL_EXTREMO:,} pesos o, si no hay precio base utilizable, mayor que {UMBRAL_EXTREMO_SIN_BASE:,}.
- **Atípico:** inconsistente o extremo. **Valor cero:** se cuenta aparte; no altera sumas.

**Resultado.**
- {efecto['n_atipico']:,} contratos atípicos ({efecto['n_inconsistente']:,} inconsistentes y {efecto['n_extremo']:,} extremos, con solapamiento) que suman {efecto['valor_atipico']:,.0f} pesos, el {100 * float(efecto['valor_atipico']) / float(efecto['valor_total']):.2f} % del valor total de gold.
- {efecto['n_cero']:,} contratos con valor 0.
- El mayor valor que queda como normal es {efecto['maximo_normal']:,.0f} pesos.
- Valor promedio por contrato: {prom_con:,.0f} pesos con los atípicos y {prom_sin:,.0f} sin ellos.
"""))


**Reglas.**
- **Inconsistente:** valor adjudicado al menos 100 veces el precio base (con precio base > 1 y valor > 0). Desde esa razón la caída natural de la distribución desaparece; es el patrón de los errores de captura (típicamente ×1.000).
- **Extremo:** valor mayor que 10,000,000,000,000 pesos o, si no hay precio base utilizable, mayor que 1,000,000,000,000.
- **Atípico:** inconsistente o extremo. **Valor cero:** se cuenta aparte; no altera sumas.

**Resultado.**
- 432 contratos atípicos (430 inconsistentes y 22 extremos, con solapamiento) que suman 174,714,776,221,732,357 pesos, el 99.80 % del valor total de gold.
- 1,527 contratos con valor 0.
- El mayor valor que queda como normal es 4,205,027,751,839 pesos.
- Valor promedio por contrato: 330,196,768,794 pesos con los atípicos y 666,167,734 sin ellos.


## T1.4 Tablas resumen

Gold ya trae el filtro de adjudicados, las columnas de entidad, contratista, valor y fecha, y las columnas `anio_adjudicacion` y `anio_mes` (T1.3). Aquí se calculan, a partir de gold con las marcas de T1.5, los agregados por **entidad**, **contratista**, **departamento** y **mes**, con el mismo patrón de agregación parcial del Punto 2 del Corte 2 (lo que en RDD es `reduceByKey` y en DataFrames es `groupBy().agg()`). Los valores salen de `valor_total_adjudicacion`, no de `precio_base`.

Cada tabla tiene una fila por clave y estas columnas de medida:
- `contratos`: todos los contratos de la clave, **incluidos** los atípicos (así la suma de `contratos` es igual a las filas de gold).
- `contratos_atipicos` y `valor_atipicos`: contratos marcados como atípicos en T1.5 (valor inconsistente con el precio base o valor extremo) y su valor.
- `contratos_valor_cero`: contratos con valor 0 (cuentan como contratos y no alteran sumas).
- `valor_total` y `valor_promedio`: calculados **sin** los atípicos. Se cumple `valor_total + valor_atipicos = valor total de gold`.

Los atributos descriptivos de cada clave (nombre, departamento) son los **más frecuentes** entre sus contratos, porque un NIT puede agrupar varias dependencias con nombres distintos.

Las tablas se guardan en `procesado/` como un único archivo Parquet cada una (son pequeñas). Las rutas están en `src/pamd/config.py`.

In [14]:
# T1.4 · Tablas resumen: una fila por clave, con los atípicos separados del valor normal
def moda(df, clave, columna):
    """Valor más frecuente de 'columna' para cada 'clave' (en empate, el primero en orden alfabético)."""
    por_clave = Window.partitionBy(clave).orderBy(F.desc("n"), F.asc(columna))
    return (
        df.filter(F.col(columna).isNotNull())
        .groupBy(clave, columna).agg(F.count("*").alias("n"))
        .withColumn("rn", F.row_number().over(por_clave))
        .filter(F.col("rn") == 1)
        .select(clave, columna)
    )

def resumir(df, claves, descriptivas=()):
    """groupBy().agg() sobre gold marcado. 'descriptivas' son atributos de la clave (se toma el más frecuente)."""
    normal = ~F.col("es_atipico")
    agregado = df.groupBy(*claves).agg(
        F.count("*").alias("contratos"),
        F.sum(F.when(normal, 1).otherwise(0)).alias("_contratos_normales"),
        F.sum(F.when(~normal, 1).otherwise(0)).alias("contratos_atipicos"),
        F.sum(F.col("es_valor_cero").cast("int")).alias("contratos_valor_cero"),
        F.sum(F.when(normal, F.col("valor_total_adjudicacion")).otherwise(0)).alias("valor_total"),
        F.sum(F.when(~normal, F.col("valor_total_adjudicacion")).otherwise(0)).alias("valor_atipicos"),
    )
    for c in descriptivas:
        agregado = agregado.join(moda(df, claves[0], c), claves[0], "left")
    promedio = F.when(F.col("_contratos_normales") > 0, F.round(F.col("valor_total") / F.col("_contratos_normales"), 2))
    return agregado.withColumn("valor_promedio", promedio).select(
        *claves, *descriptivas, "contratos", "contratos_atipicos", "contratos_valor_cero",
        "valor_total", "valor_atipicos", "valor_promedio",
    )

# Los contratos sin departamento no se ocultan: forman la categoría 'SIN DEPARTAMENTO'
SIN_DEPARTAMENTO = "SIN DEPARTAMENTO"
gold_con_depto = gold_marcado.withColumn(
    "departamento",
    F.when(F.trim("departamento_entidad") != "", F.trim("departamento_entidad")).otherwise(F.lit(SIN_DEPARTAMENTO)),
)

CLAVES_RESUMEN = {
    "entidad": ["id_entidad"],
    "contratista": ["id_contratista"],
    "departamento": ["departamento"],
    "mes": ["anio_mes"],
}
resumen_entidad_df = resumir(gold_marcado, ["id_entidad"], ["nombre_entidad", "departamento_entidad"])
resumen_contratista_df = resumir(gold_marcado, ["id_contratista"], ["nombre_contratista", "departamento_contratista"])
resumen_departamento_df = resumir(gold_con_depto, ["departamento"])
resumen_mes_df = resumir(gold_marcado, ["anio_mes"], ["anio_adjudicacion"])

### Escritura de las tablas resumen

`RECONSTRUIR_RESUMENES = True` recalcula y escribe las cuatro tablas en `procesado/`; con `False` se reutiliza lo que ya está en MinIO.

In [15]:
# T1.4 · Escritura de las tablas resumen (un archivo Parquet por tabla)
# True la primera vez (escribe las cuatro tablas); False reutiliza lo que ya está en MinIO.
RECONSTRUIR_RESUMENES = False

RUTAS_RESUMEN = {
    "entidad": PROCESADO_ENTIDAD_PARQUET,
    "contratista": PROCESADO_CONTRATISTA_PARQUET,
    "departamento": PROCESADO_DEPARTAMENTO_PARQUET,
    "mes": PROCESADO_MES_PARQUET,
}

if RECONSTRUIR_RESUMENES:
    por_escribir = {
        "entidad": resumen_entidad_df,
        "contratista": resumen_contratista_df,
        "departamento": resumen_departamento_df,
        "mes": resumen_mes_df,
    }
    for nombre, df in por_escribir.items():
        df.coalesce(1).write.mode(WRITE_MODE).option("compression", "snappy").parquet(RUTAS_RESUMEN[nombre])

resumenes = {nombre: spark.read.parquet(ruta) for nombre, ruta in RUTAS_RESUMEN.items()}

### Esquema de las tablas resumen

In [16]:
# T1.4 · Esquema documentado de cada tabla (tipo y significado de cada columna)
significado_resumen = {
    "id_entidad": "Identificador de la entidad contratante (misma regla que gold, con los NIT de dígito de verificación pegado unificados). Clave de la tabla por entidad.",
    "nombre_entidad": "Nombre más frecuente entre los contratos de la entidad (un NIT puede agrupar varias dependencias).",
    "departamento_entidad": "Departamento más frecuente entre los contratos de la entidad.",
    "id_contratista": "Identificador del contratista (NIT, luego código, luego nombre). Clave de la tabla por contratista.",
    "nombre_contratista": "Nombre más frecuente entre los contratos del contratista.",
    "departamento_contratista": "Departamento más frecuente del contratista (puede ser nulo).",
    "departamento": "Departamento de la entidad contratante; los contratos sin departamento van a 'SIN DEPARTAMENTO'. Clave de la tabla por departamento.",
    "anio_mes": "Año y mes de adjudicación, 'AAAA-MM'. Clave de la tabla por mes.",
    "anio_adjudicacion": "Año de adjudicación del mes.",
    "contratos": "Número de contratos adjudicados de la clave, con los atípicos incluidos.",
    "contratos_atipicos": "Contratos de la clave marcados como atípicos en T1.5 (valor inconsistente con el precio base o extremo).",
    "contratos_valor_cero": "Contratos de la clave con valor 0.",
    "valor_total": "Suma de valor_total_adjudicacion sin los atípicos, en pesos.",
    "valor_atipicos": "Suma de valor_total_adjudicacion de los contratos atípicos, en pesos.",
    "valor_promedio": "valor_total dividido por los contratos que no son atípicos; nulo si todos lo son.",
}
for nombre, df in resumenes.items():
    esquema = pd.DataFrame(
        [(f.name, f.dataType.simpleString(), significado_resumen.get(f.name, "")) for f in df.schema.fields],
        columns=["columna", "tipo", "significado"],
    )
    assert (esquema["significado"] != "").all(), f"Hay columnas sin documentar en la tabla por {nombre}"
    display(Markdown(f"**Tabla por {nombre}** (`{RUTAS_RESUMEN[nombre]}`)"))
    display(esquema)

**Tabla por entidad** (`s3a://rawdata/procesado/adjudicaciones_por_entidad`)

,columna,tipo,significado
0,id_entidad,string,Identificador de la entidad contratante (misma...
1,nombre_entidad,string,Nombre más frecuente entre los contratos de la...
2,departamento_entidad,string,Departamento más frecuente entre los contratos...
3,contratos,bigint,"Número de contratos adjudicados de la clave, c..."
4,contratos_atipicos,bigint,Contratos de la clave marcados como atípicos e...
5,contratos_valor_cero,bigint,Contratos de la clave con valor 0.
6,valor_total,"decimal(30,2)",Suma de valor_total_adjudicacion sin los atípi...
7,valor_atipicos,"decimal(30,2)",Suma de valor_total_adjudicacion de los contra...
8,valor_promedio,"decimal(31,2)",valor_total dividido por los contratos que no ...


**Tabla por contratista** (`s3a://rawdata/procesado/adjudicaciones_por_contratista`)

,columna,tipo,significado
0,id_contratista,string,"Identificador del contratista (NIT, luego códi..."
1,nombre_contratista,string,Nombre más frecuente entre los contratos del c...
2,departamento_contratista,string,Departamento más frecuente del contratista (pu...
3,contratos,bigint,"Número de contratos adjudicados de la clave, c..."
4,contratos_atipicos,bigint,Contratos de la clave marcados como atípicos e...
5,contratos_valor_cero,bigint,Contratos de la clave con valor 0.
6,valor_total,"decimal(30,2)",Suma de valor_total_adjudicacion sin los atípi...
7,valor_atipicos,"decimal(30,2)",Suma de valor_total_adjudicacion de los contra...
8,valor_promedio,"decimal(31,2)",valor_total dividido por los contratos que no ...


**Tabla por departamento** (`s3a://rawdata/procesado/adjudicaciones_por_departamento`)

,columna,tipo,significado
0,departamento,string,Departamento de la entidad contratante; los co...
1,contratos,bigint,"Número de contratos adjudicados de la clave, c..."
2,contratos_atipicos,bigint,Contratos de la clave marcados como atípicos e...
3,contratos_valor_cero,bigint,Contratos de la clave con valor 0.
4,valor_total,"decimal(30,2)",Suma de valor_total_adjudicacion sin los atípi...
5,valor_atipicos,"decimal(30,2)",Suma de valor_total_adjudicacion de los contra...
6,valor_promedio,"decimal(31,2)",valor_total dividido por los contratos que no ...


**Tabla por mes** (`s3a://rawdata/procesado/adjudicaciones_por_mes`)

,columna,tipo,significado
0,anio_mes,string,"Año y mes de adjudicación, 'AAAA-MM'. Clave de..."
1,anio_adjudicacion,int,Año de adjudicación del mes.
2,contratos,bigint,"Número de contratos adjudicados de la clave, c..."
3,contratos_atipicos,bigint,Contratos de la clave marcados como atípicos e...
4,contratos_valor_cero,bigint,Contratos de la clave con valor 0.
5,valor_total,"decimal(30,2)",Suma de valor_total_adjudicacion sin los atípi...
6,valor_atipicos,"decimal(30,2)",Suma de valor_total_adjudicacion de los contra...
7,valor_promedio,"decimal(31,2)",valor_total dividido por los contratos que no ...


### Criterios de aceptación

1. Cada tabla tiene su esquema descrito (celda anterior).
2. La suma de cada resumen es igual a la de gold: contratos, contratos atípicos, contratos con valor cero, y `valor_total + valor_atipicos` igual al valor total de gold (con `assert`, en pesos exactos).
3. Cada tabla tiene una sola fila por clave y ninguna clave nula.

In [17]:
# T1.4 · Criterios de aceptación: cada resumen suma exactamente lo mismo que gold
tot_gold = gold.agg(F.count("*").alias("contratos"), F.sum("valor_total_adjudicacion").alias("valor")).first()
tot_marcas = gold_marcado.agg(
    F.sum(F.col("es_atipico").cast("int")).alias("atipicos"),
    F.sum(F.col("es_valor_cero").cast("int")).alias("ceros"),
).first()

filas_resumen = []
for nombre, df in resumenes.items():
    r = df.agg(
        F.count("*").alias("filas"),
        F.sum("contratos").alias("contratos"),
        F.sum("contratos_atipicos").alias("atipicos"),
        F.sum("contratos_valor_cero").alias("ceros"),
        F.sum("valor_total").alias("valor_total"),
        F.sum("valor_atipicos").alias("valor_atipicos"),
    ).first()
    clave = CLAVES_RESUMEN[nombre]
    assert r["contratos"] == tot_gold["contratos"], f"{nombre}: la suma de contratos no es la de gold"
    assert r["atipicos"] == tot_marcas["atipicos"], f"{nombre}: los contratos atípicos no suman los marcados"
    assert r["ceros"] == tot_marcas["ceros"], f"{nombre}: los contratos con valor cero no suman los marcados"
    assert r["valor_total"] + r["valor_atipicos"] == tot_gold["valor"], f"{nombre}: valor_total + valor_atipicos no es el valor de gold"
    assert df.select(*clave).distinct().count() == r["filas"], f"{nombre}: hay claves repetidas"
    assert df.filter(F.col(clave[0]).isNull()).count() == 0, f"{nombre}: hay claves nulas"
    filas_resumen.append({"tabla": nombre, "filas": r["filas"], "contratos": r["contratos"], "contratos atípicos": r["atipicos"],
                          "valor_total": r["valor_total"], "valor_atipicos": r["valor_atipicos"]})

display(pd.DataFrame(filas_resumen).style.format({"filas": "{:,}", "contratos": "{:,}", "contratos atípicos": "{:,}",
                                                    "valor_total": "{:,.0f}", "valor_atipicos": "{:,.0f}"}))
print(f"Gold: {tot_gold['contratos']:,} contratos, {tot_marcas['atipicos']:,} atípicos, {tot_gold['valor']:,.2f} pesos.")
print("Criterios de aceptación de T1.4: OK")

# Vista rápida: las 5 entidades con más valor (sin atípicos) y el valor por departamento
display(pd.DataFrame([r.asDict() for r in resumenes["entidad"].orderBy(F.desc("valor_total")).limit(5).collect()]))
display(pd.DataFrame([r.asDict() for r in resumenes["departamento"].orderBy(F.desc("valor_total")).limit(8).collect()]))

,tabla,filas,contratos,contratos atípicos,valor_total,valor_atipicos
0,entidad,"4,050","530,192",432,"352,909,018,925,397","174,714,776,221,732,357"
1,contratista,"99,606","530,192",432,"352,909,018,925,397","174,714,776,221,732,357"
2,departamento,34,"530,192",432,"352,909,018,925,397","174,714,776,221,732,357"
3,mes,136,"530,192",432,"352,909,018,925,397","174,714,776,221,732,357"


Gold: 530,192 contratos, 432 atípicos, 175,067,685,240,657,754.00 pesos.
Criterios de aceptación de T1.4: OK


,id_entidad,nombre_entidad,departamento_entidad,contratos,contratos_atipicos,contratos_valor_cero,valor_total,valor_atipicos,valor_promedio
0,NIT:800215807,INVIAS,Distrito Capital de Bogotá,5285,2,5,22256198017874.00,82194585809.00,4212795384.80
1,NIT:899999061,(Secretaría Distrital de Integración Social),Distrito Capital de Bogotá,18126,6,128,19436497397090.00,11518692965.00,1072654381.74
2,NIT:890905211,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNO...,Antioquia,4879,3,10,15980884831288.00,422330658229.00,3277457922.74
3,NIT:899999081,INSTITUTO DE DESARROLLO URBANO,Distrito Capital de Bogotá,864,1,4,11453591009173.00,3500000000.00,13271831992.09
4,NIT:900475780,UNP,Distrito Capital de Bogotá,615,0,3,8226967368610.00,0.00,13377182713.19


,departamento,contratos,contratos_atipicos,contratos_valor_cero,valor_total,valor_atipicos,valor_promedio
0,Distrito Capital de Bogotá,150188,169,844,189139613976302.00,6301142013668195.00,1260771062.17
1,Antioquia,59982,43,132,38696546390939.00,98726539165109588.00,645598798.63
2,Valle del Cauca,34119,23,74,15076714612541.00,24194257526330090.00,442184262.45
3,Cundinamarca,29126,23,52,10768458248252.00,13797170600029.00,370011966.06
4,Bolívar,13311,10,15,9487438647058.00,8861080000023.00,713287621.01
5,Santander,24481,23,47,9363747585156.00,8815438682053354.00,382850093.43
6,Atlántico,10683,4,10,7386003332930.00,642218546676300.00,691638105.90
7,Boyacá,30007,21,20,6696680134674.00,11894834438238933.00,223326890.37


### Para qué sirve cada tabla

| Tabla | Ruta en MinIO | Módulo del dashboard (Punto 4) |
|---|---|---|
| por mes | `procesado/adjudicaciones_por_mes` | volumen contratado en el tiempo |
| por entidad | `procesado/adjudicaciones_por_entidad` | ranking de entidades y filtro por entidad |
| por contratista | `procesado/adjudicaciones_por_contratista` | ranking de contratistas |
| por departamento | `procesado/adjudicaciones_por_departamento` | filtro por región y concentración geográfica |

Estas son las tablas globales. T4.1 añadirá las series mensuales por departamento y entidad que necesitan los filtros combinados del dashboard. El streaming del Punto 2 y el dashboard deben usar `marcar_atipicos` y `unificar_nit_entidad` de `src/pamd/calidad.py` para que sus cifras coincidan con estas.

## T1.6 Articulación de los cortes 1 y 2 y linaje de datos

El Corte 3 no rehace los cortes anteriores: se apoya en ellos. Del Corte 1 conserva la separación entre almacenamiento y cómputo (MinIO guarda, Spark calcula) y el bronze en Parquet, con su cuarentena de 427 registros inválidos; el CSV de SECOP II se lee una sola vez. Del Corte 2 reutiliza la agregación parcial (`groupBy().agg()` en DataFrames, el equivalente de `reduceByKey`, que movió 5,2 veces menos datos en el shuffle que `groupByKey`) para construir las tablas resumen; la regla de identificador de contratista del PageRank entidad → contratista (NIT, luego código, luego nombre); y la lección del sesgo: apenas el 2,5 % de las entidades concentra la mitad de los registros, así que los rankings dependen de pocos actores. Lo nuevo del Corte 3 es limpiar lo que el bronze conserva: silver deja una fila por proceso y adjudicación, gold deja solo los contratos adjudicados con fecha, entidad y contratista, y las tablas de `procesado/` resumen gold por entidad, contratista, departamento y mes, con los valores atípicos marcados y separados del valor normal. Sobre gold se construirán el flujo con ventanas, los resultados idempotentes y el dashboard. Las cifras del Corte 2 se calcularon sobre el bronze, que conserva las repeticiones (9.099.432 registros), y las del Corte 3 sobre gold, ya deduplicado (530.192 contratos): no son comparables entre sí.

![Linaje de datos](../docs/linaje_datos.png)

El diagrama está en `docs/linaje_datos.svg` (y `.png` para la presentación), y las tablas de rutas y de reutilización por corte en `docs/articulacion_cortes.md`.

| Corte | Qué se reutiliza | Dónde aparece en el Corte 3 |
|---|---|---|
| 1 | MinIO y Parquet; bronze y cuarentena | entrada de silver; rutas compartidas en `src/pamd/config.py` |
| 1 | CSV leído una sola vez | el CSV solo se vuelve a leer al reconstruir el bronze |
| 2 | agregación parcial (`reduceByKey`, en DataFrames `groupBy().agg()`) | tablas resumen de T1.4 y agregación por ventana del Punto 2 |
| 2 | regla de identificador de contratista (NIT, código, nombre) | columna `id_contratista` de gold |
| 2 | sesgo de datos | lectura de los rankings: pocas entidades concentran la mitad de los registros |
| 2 | PageRank entidad → contratista | opcional: tercera lectura del ranking del dashboard (T4.5) |

Las cifras del Corte 2 (PageRank, `groupByKey` frente a `reduceByKey`, sesgo) se calcularon sobre el bronze, que conserva las repeticiones. Las del Corte 3 salen de gold, ya deduplicado. Por eso, por ejemplo, el bronze tiene 872.756 filas con `adjudicado = True` y gold tiene 530.192 contratos: no son comparables entre sí.

### Las flechas del diagrama son rutas reales

La celda siguiente comprueba que cada carpeta del diagrama existe en MinIO y cuenta sus filas. Las cifras del diagrama y de `docs/articulacion_cortes.md` son las de esta tabla.

In [2]:
# T1.6 · Cada carpeta del diagrama de linaje existe en MinIO; se cuentan sus filas
import pandas as pd

LINAJE = [
    ("CSV de SECOP II", INPUT_CSV, False),
    ("Staging", STAGING_PARQUET, True),
    ("Bronze", BRONZE_PARQUET_UNPARTITIONED, True),
    ("Bronze particionado", BRONZE_PARQUET, True),
    ("Cuarentena", QUARANTINE_PARQUET, True),
    ("Silver", SILVER_PARQUET, True),
    ("Silver descartados", SILVER_DESCARTADOS_PARQUET, True),
    ("Gold", GOLD_PARQUET, True),
    ("Gold excluidos", GOLD_EXCLUIDOS_PARQUET, True),
    ("Resumen por entidad", PROCESADO_ENTIDAD_PARQUET, True),
    ("Resumen por contratista", PROCESADO_CONTRATISTA_PARQUET, True),
    ("Resumen por departamento", PROCESADO_DEPARTAMENTO_PARQUET, True),
    ("Resumen por mes", PROCESADO_MES_PARQUET, True),
]

def existe(ruta):
    jvm = spark.sparkContext._jvm
    path = jvm.org.apache.hadoop.fs.Path(ruta)
    return path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration()).exists(path)

filas_linaje, conteos = [], {}
for nombre, ruta, contar in LINAJE:
    assert existe(ruta), f"No existe la ruta del linaje: {ruta}"
    n = spark.read.parquet(ruta).count() if contar else None  # el CSV no se cuenta: su lectura es lenta
    conteos[nombre] = n
    filas_linaje.append({"etapa": nombre, "ruta": ruta, "filas": n})

display(pd.DataFrame(filas_linaje).style.format({"filas": "{:,.0f}"}, na_rep="no se cuenta"))

# Las cifras del linaje cuadran entre sí
assert conteos["Staging"] == conteos["Bronze"] + conteos["Cuarentena"], "Staging != bronze + cuarentena"
assert conteos["Bronze"] == conteos["Bronze particionado"], "El bronze particionado no tiene las mismas filas"
assert conteos["Bronze"] == conteos["Silver"] + conteos["Silver descartados"], "Bronze != silver + descartados"
print("Linaje de datos: todas las rutas existen y las cifras cuadran")

,etapa,ruta,filas
0,CSV de SECOP II,s3a://rawdata/secop/,no se cuenta
1,Staging,s3a://rawdata/staging/secop/con_calidad,"9,099,859"
2,Bronze,s3a://rawdata/bronze/secop/contratos,"9,099,432"
3,Bronze particionado,s3a://rawdata/bronze/secop/contratos_particionado,"9,099,432"
4,Cuarentena,s3a://rawdata/cuarentena/secop,427
5,Silver,s3a://rawdata/silver/secop/contratos,"8,694,126"
6,Silver descartados,s3a://rawdata/silver/secop/descartados,"405,306"
7,Gold,s3a://rawdata/gold/secop/adjudicaciones,"530,192"
8,Gold excluidos,s3a://rawdata/gold/secop/excluidos,"17,226"
9,Resumen por entidad,s3a://rawdata/procesado/adjudicaciones_por_entidad,"4,050"


Linaje de datos: todas las rutas existen y las cifras cuadran
